In [7]:
%pip install pandas numpy matplotlib seaborn scikit-learn pyarrow

Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
import pickle
import pyarrow.parquet as pq
import glob
import pyarrow as pa
import os

In [3]:
Xt=["usr", "nnumr", "cnumr", "elpl", "jobenv_req", "pri", "mszl", "freq_req"]
Yt=["duration"]

In [10]:
target_columns = ["usr", "nnumr", "cnumr", "elpl", "jobenv_req", "pri", "mszl", "freq_req", "duration"]

file_paths = [
    "21_05.parquet",
    "21_10.parquet",
    "22_06.parquet",
    "23_12.parquet",
    "24_02.parquet"
]

dfs = []

for path in file_paths:
    print(f"\n {path} kontrol ediliyor...")
    try:
        table = pq.read_table(path)
        available_cols = table.schema.names
        print(f" Sütunlar: {available_cols}")

        valid_cols = [col for col in target_columns if col in available_cols]
        if not valid_cols:
            print(f" Uygun sütun yok, atlanıyor.")
            continue

        df = table.to_pandas()           
        df = df[valid_cols]              
        df = df.reindex(columns=target_columns)  
        dfs.append(df)
        print(f" Eklendi: {df.shape[0]} satır, {len(valid_cols)} sütun.")

    except Exception as e:
        print(f" {path} → Hata: {type(e).__name__}: {e}")


if dfs:
    df_combined = pd.concat(dfs, ignore_index=True)
    print(f"\n Birleştirme tamam: {df_combined.shape[0]} satır, {df_combined.shape[1]} sütun.")
    print(df_combined.head())

    
else:
    print(" Uygun veri içeren dosya bulunamadı.")



 21_05.parquet kontrol ediliyor...
 Sütunlar: ['jid', 'usr', 'jnam', 'cnumr', 'cnumat', 'cnumut', 'nnumr', 'adt', 'qdt', 'schedsdt', 'deldt', 'ec', 'elpl', 'sdt', 'edt', 'nnuma', 'idle_time_ave', 'nnumu', 'perf1', 'perf2', 'perf3', 'perf4', 'perf5', 'perf6', 'mszl', 'pri', 'econ', 'avgpcon', 'minpcon', 'maxpcon', 'msza', 'mmszu', 'uctmut', 'sctmut', 'usctmut', 'jobenv_req', 'freq_req', 'freq_alloc', 'flops', 'mbwidth', 'opint', 'pclass', 'embedding', 'exit state', 'duration']
 Eklendi: 595314 satır, 9 sütun.

 21_10.parquet kontrol ediliyor...
 Sütunlar: ['jid', 'usr', 'jnam', 'cnumr', 'cnumat', 'cnumut', 'nnumr', 'adt', 'qdt', 'schedsdt', 'deldt', 'ec', 'elpl', 'sdt', 'edt', 'nnuma', 'idle_time_ave', 'nnumu', 'perf1', 'perf2', 'perf3', 'perf4', 'perf5', 'perf6', 'mszl', 'pri', 'econ', 'avgpcon', 'minpcon', 'maxpcon', 'msza', 'mmszu', 'uctmut', 'sctmut', 'usctmut', 'jobenv_req', 'freq_req', 'freq_alloc', 'flops', 'mbwidth', 'opint', 'pclass', 'embedding', 'exit state', 'duration']
 Ek

In [11]:
df_combined.to_parquet("F_dataset.parquet", index=False)

In [12]:
df=pd.read_parquet("F_dataset.parquet")
df["duration"] = df["duration"] / 60
df=df[df["duration"] > 1]
df.to_parquet("F_dataset.parquet", index=False)

In [4]:
df=pd.read_parquet("F_dataset.parquet")
df["elpl"] = df["elpl"] / 60
df.to_parquet("F_dataset.parquet", index=False)

In [4]:
N_L = 5000
chunk_size = 50_000
xt_path = "Xt.parquet"
yt_path = "Yt.parquet"

pf_xt = pq.ParquetFile(xt_path)
rows_read = 0
dfs = []

for rg in range(pf_xt.num_row_groups):
    table = pf_xt.read_row_group(rg)
    df = table.to_pandas()
    if rows_read + len(df) < N_L:
        dfs.append(df)
        rows_read += len(df)
    else:
        remaining = N_L - rows_read
        dfs.append(df.iloc[:remaining])
        break

X_t_L_df = pd.concat(dfs, ignore_index=True)
X_t_L_df.to_parquet("Xt_L.parquet", index=False)

with pq.ParquetWriter("Xt_U.parquet", schema=pf_xt.schema_arrow,
                      compression="snappy", use_dictionary=False) as writer:
    written = 0
    for rg in range(pf_xt.num_row_groups):
        table = pf_xt.read_row_group(rg)
        df = table.to_pandas()

        if written + len(df) <= N_L:
            written += len(df)
            continue
        elif written < N_L:
            start = N_L - written
            df = df.iloc[start:]

        for i in range(0, len(df), chunk_size):
            chunk_df = df.iloc[i:i+chunk_size]
            table = pa.Table.from_pandas(chunk_df, preserve_index=False)
            writer.write_table(table)
        written += len(df)

Y_t_df = pd.read_parquet(yt_path)
Y_t_L_df = Y_t_df.iloc[:N_L]
Y_t_L_df.columns = ["duration"]  

Y_t_L_df.to_parquet("Yt_L.parquet", index=False)

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
import optuna
import copy
from sklearn.metrics import mean_squared_error

In [3]:
Xs = pd.read_parquet("Xs.parquet")
Ys = pd.read_parquet("Ys.parquet")["run_time"]
df = Xs.copy()
df["run_time"] = Ys
df_clean = df.dropna()
Xs_clean = df_clean.drop(columns=["run_time"])
Ys_clean = df_clean["run_time"]

X_train, X_val, y_train, y_val = train_test_split(
    Xs_clean, Ys_clean, test_size=0.2, random_state=42
)

In [4]:
X_train_tensor = torch.tensor(X_train.values, dtype=torch.float32)
X_val_tensor   = torch.tensor(X_val.values,   dtype=torch.float32)

y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1)
y_val_tensor   = torch.tensor(y_val.values,   dtype=torch.float32).unsqueeze(1)

In [5]:
def objective(trial):
    hidden1 = trial.suggest_int("hidden1", 64, 256)
    hidden2 = trial.suggest_int("hidden2", 32, 128)
    hidden3 = trial.suggest_int("hidden3", 16, 64)
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])
    n_epochs = trial.suggest_int("n_epochs", 50, 200)

    model = nn.Sequential(
        nn.Linear(X_train_tensor.shape[1], hidden1),
        nn.ReLU(),
        nn.Linear(hidden1, hidden2),
        nn.ReLU(),
        nn.Linear(hidden2, hidden3),
        nn.ReLU(),
        nn.Linear(hidden3, 1)
    )

    loss_fn = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    batch_start = torch.arange(0, len(X_train_tensor), batch_size)

    model.eval()
    with torch.no_grad():
        init_pred = model(X_val_tensor)
        best_mse = loss_fn(init_pred, y_val_tensor).item()

        if np.isnan(best_mse) or np.isinf(best_mse):
            return float("inf")

        best_weights = copy.deepcopy(model.state_dict())

    for epoch in range(n_epochs):
        model.train()
        for start in batch_start:
            X_batch = X_train_tensor[start:start+batch_size]
            y_batch = y_train_tensor[start:start+batch_size]

            y_pred = model(X_batch)
            loss = loss_fn(y_pred, y_batch)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            y_pred = model(X_val_tensor)
            mse = loss_fn(y_pred, y_val_tensor).item()

            if not np.isnan(mse) and mse < best_mse:
                best_mse = mse
                best_weights = copy.deepcopy(model.state_dict())

    if best_weights is not None:
        model.load_state_dict(best_weights)

    if np.isnan(best_mse) or np.isinf(best_mse):
        return float("inf")

    return best_mse


In [19]:
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=30, timeout=600)

print("Best parameters: ", study.best_params)
print("Least MSE (L_s):", study.best_value)


[I 2025-09-08 16:03:23,318] A new study created in memory with name: no-name-e57f724d-11c5-4b9a-82c1-42736ee54eb3
[I 2025-09-08 16:03:24,285] Trial 0 finished with value: 4201.41552734375 and parameters: {'hidden1': 217, 'hidden2': 69, 'hidden3': 63, 'lr': 0.0009844295776420973, 'batch_size': 128, 'n_epochs': 93}. Best is trial 0 with value: 4201.41552734375.
[I 2025-09-08 16:03:29,684] Trial 1 finished with value: 3522.37353515625 and parameters: {'hidden1': 164, 'hidden2': 86, 'hidden3': 47, 'lr': 0.004837443508686495, 'batch_size': 32, 'n_epochs': 183}. Best is trial 1 with value: 3522.37353515625.
[I 2025-09-08 16:03:33,463] Trial 2 finished with value: 3934.74755859375 and parameters: {'hidden1': 198, 'hidden2': 99, 'hidden3': 59, 'lr': 0.0005196428861550251, 'batch_size': 32, 'n_epochs': 122}. Best is trial 1 with value: 3522.37353515625.
[I 2025-09-08 16:03:36,983] Trial 3 finished with value: 3622.1474609375 and parameters: {'hidden1': 212, 'hidden2': 45, 'hidden3': 60, 'lr': 0

Best parameters:  {'hidden1': 147, 'hidden2': 53, 'hidden3': 16, 'lr': 0.007343613794066759, 'batch_size': 64, 'n_epochs': 172}
Least MSE (L_s): 3354.9619140625


In [6]:
best_params = {
    'hidden1': 147,
    'hidden2': 53,
    'hidden3': 16,
    'lr': 0.007343613794066759,
    'batch_size': 64,
    'n_epochs': 172
}

In [7]:
model_fs = nn.Sequential(
    nn.Linear(X_train_tensor.shape[1], best_params['hidden1']),
    nn.ReLU(),
    nn.Linear(best_params['hidden1'], best_params['hidden2']),
    nn.ReLU(),
    nn.Linear(best_params['hidden2'], best_params['hidden3']),
    nn.ReLU(),
    nn.Linear(best_params['hidden3'], 1)
)

In [8]:
loss_fn = nn.MSELoss()
optimizer = optim.Adam(model_fs.parameters(), lr=best_params['lr'])

In [9]:
batch_size = best_params['batch_size']
n_epochs = best_params['n_epochs']
batch_start = torch.arange(0, len(X_train_tensor), batch_size)

best_mse = float("inf")
best_weights = None

In [10]:
for epoch in range(n_epochs):
    model_fs.train()
    for start in batch_start:
        X_batch = X_train_tensor[start:start+batch_size]
        y_batch = y_train_tensor[start:start+batch_size]

        y_pred = model_fs(X_batch)
        loss = loss_fn(y_pred, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    model_fs.eval()
    with torch.no_grad():
        y_val_pred = model_fs(X_val_tensor)
        mse = loss_fn(y_val_pred, y_val_tensor).item()

        if mse < best_mse:
            best_mse = mse
            best_weights = copy.deepcopy(model_fs.state_dict())


In [11]:
if best_weights is not None:
    model_fs.load_state_dict(best_weights)

In [12]:
print(f"Final FS model MSE (L_s): {best_mse}")

Final FS model MSE (L_s): 3515.872314453125


In [27]:
torch.save(model_fs.state_dict(), "fs_model.pth")

In [13]:
Xt_L = pd.read_parquet("Xt_L.parquet")
Yt_L = pd.read_parquet("Yt_L.parquet")["duration"]  

Xtl_tensor = torch.tensor(Xt_L.values, dtype=torch.float32)
Ytl_tensor = torch.tensor(Yt_L.values.reshape(-1, 1), dtype=torch.float32)


In [18]:
Xs = pd.read_parquet("Xs.parquet")
input_dim = Xs.shape[1]
Xs.shape[1]

23

In [19]:
fs_model = nn.Sequential(
    nn.Linear(23, 147),  
    nn.ReLU(),
    nn.Linear(147, 53),
    nn.ReLU(),
    nn.Linear(53, 16),
    nn.ReLU(),
    nn.Linear(16, 1)
)

In [20]:
fs_model.load_state_dict(torch.load("fs_model.pth"))
fs_model.eval()

Sequential(
  (0): Linear(in_features=23, out_features=147, bias=True)
  (1): ReLU()
  (2): Linear(in_features=147, out_features=53, bias=True)
  (3): ReLU()
  (4): Linear(in_features=53, out_features=16, bias=True)
  (5): ReLU()
  (6): Linear(in_features=16, out_features=1, bias=True)
)

In [21]:
output_dim=23

In [22]:
def create_ma_model(input_dim, hidden_dim, output_dim):
    return nn.Sequential(
        nn.Linear(input_dim, hidden_dim),
        nn.ReLU(),
        nn.Linear(hidden_dim, hidden_dim),
        nn.ReLU(),
        nn.Linear(hidden_dim, output_dim)  
    )

In [23]:
import torch.nn.functional as F
def objective(trial):
    hidden_dim = trial.suggest_int("hidden_dim", 16, 128)
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
    num_epochs = trial.suggest_int("epochs", 100, 300)

    ma_model = create_ma_model(Xtl_tensor.shape[1], hidden_dim, 23)
    optimizer = torch.optim.Adam(ma_model.parameters(), lr=lr, weight_decay=weight_decay)

    for epoch in range(num_epochs):
        ma_model.train()
        optimizer.zero_grad()

        Ztl = ma_model(Xtl_tensor)              
        Y_pred = fs_model(Ztl)                   
        loss = F.mse_loss(Y_pred, Ytl_tensor)    

        loss.backward()
        optimizer.step()

    return loss.item()


In [24]:
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=20, timeout=600)

print("Best hyperparameters:", study.best_trial.params)


[I 2025-09-09 14:57:18,477] A new study created in memory with name: no-name-88b317ba-b940-43d7-9989-b4a4884b24ef
[I 2025-09-09 14:57:26,603] Trial 0 finished with value: 5110.56298828125 and parameters: {'hidden_dim': 80, 'lr': 0.00029800446138103346, 'weight_decay': 1.7156091486179985e-06, 'epochs': 231}. Best is trial 0 with value: 5110.56298828125.
[I 2025-09-09 14:57:30,050] Trial 1 finished with value: 4582.90478515625 and parameters: {'hidden_dim': 39, 'lr': 0.0040669200502828394, 'weight_decay': 0.0001980411174133373, 'epochs': 138}. Best is trial 1 with value: 4582.90478515625.
[I 2025-09-09 14:57:33,919] Trial 2 finished with value: 4234.43408203125 and parameters: {'hidden_dim': 34, 'lr': 0.008317365968919858, 'weight_decay': 0.00012823072170754652, 'epochs': 166}. Best is trial 2 with value: 4234.43408203125.
[I 2025-09-09 14:57:37,729] Trial 3 finished with value: 21792.3203125 and parameters: {'hidden_dim': 17, 'lr': 0.00037543743149644654, 'weight_decay': 4.3941702486753

Best hyperparameters: {'hidden_dim': 109, 'lr': 0.009326048135745422, 'weight_decay': 0.0007671508137271531, 'epochs': 297}


In [25]:
best = study.best_trial.params

ma_model = create_ma_model(Xtl_tensor.shape[1], best["hidden_dim"], 23)
optimizer = torch.optim.Adam(ma_model.parameters(), lr=best["lr"], weight_decay=best["weight_decay"])

for epoch in range(best["epochs"]):
    ma_model.train()
    optimizer.zero_grad()

    Ztl = ma_model(Xtl_tensor)
    Y_pred = fs_model(Ztl)
    loss = F.mse_loss(Y_pred, Ytl_tensor)

    loss.backward()
    optimizer.step()

print("Final loss:", loss.item())


Final loss: 4184.89697265625


In [26]:
torch.save(ma_model.state_dict(), "ma_model.pth")

In [27]:
with torch.no_grad():
    Ztl_tensor = ma_model(Xtl_tensor)         
    Ytl_pred_ma = fs_model(Ztl_tensor)        

MR_input = torch.cat([Xtl_tensor, Ytl_pred_ma], dim=1)


In [32]:
def objective_fr(trial):
    hidden_dim = trial.suggest_int("hidden_dim", 16, 128)
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
    num_epochs = trial.suggest_int("epochs", 100, 300)

    fr_model = nn.Sequential(
        nn.Linear(MR_input.shape[1], hidden_dim),
        nn.ReLU(),
        nn.Linear(hidden_dim, 1)  
    )

    optimizer = torch.optim.Adam(fr_model.parameters(), lr=lr, weight_decay=weight_decay)

    for epoch in range(num_epochs):
        fr_model.train()
        optimizer.zero_grad()

        YtL_pred_fr = fr_model(MR_input)  
        loss = F.mse_loss(YtL_pred_fr, Ytl_tensor)  

        loss.backward()
        optimizer.step()

    return loss.item()

In [34]:
study = optuna.create_study(direction="minimize")
study.optimize(objective_fr, n_trials=20, timeout=600)

print("Best hyperparameters:", study.best_trial.params)

[I 2025-09-09 15:54:04,401] A new study created in memory with name: no-name-600a2cc2-6a27-4b49-a4a6-a48eab8de87c
[I 2025-09-09 15:54:10,769] Trial 0 finished with value: 31814.7578125 and parameters: {'hidden_dim': 89, 'lr': 0.00017811375168042023, 'weight_decay': 3.0282546108572594e-06, 'epochs': 252}. Best is trial 0 with value: 31814.7578125.
[I 2025-09-09 15:54:15,239] Trial 1 finished with value: 45658.8359375 and parameters: {'hidden_dim': 84, 'lr': 0.00016671743951126255, 'weight_decay': 1.968116492807982e-06, 'epochs': 190}. Best is trial 0 with value: 31814.7578125.
[I 2025-09-09 15:54:17,180] Trial 2 finished with value: 54652.96484375 and parameters: {'hidden_dim': 18, 'lr': 0.0001803768229460245, 'weight_decay': 4.77461262463045e-06, 'epochs': 220}. Best is trial 0 with value: 31814.7578125.
[I 2025-09-09 15:54:20,109] Trial 3 finished with value: 4785.35400390625 and parameters: {'hidden_dim': 74, 'lr': 0.0009030148331649266, 'weight_decay': 1.5890803800620767e-06, 'epoch

Best hyperparameters: {'hidden_dim': 126, 'lr': 0.009569106605807188, 'weight_decay': 0.0006414639131483594, 'epochs': 293}


In [35]:
hidden_dim = 126
lr = 0.009569106605807188
weight_decay = 0.0006414639131483594
num_epochs = 293

In [36]:
fr_model = nn.Sequential(
    nn.Linear(MR_input.shape[1], hidden_dim),
    nn.ReLU(),
    nn.Linear(hidden_dim, 1)
)

optimizer = torch.optim.Adam(fr_model.parameters(), lr=lr, weight_decay=weight_decay)

for epoch in range(num_epochs):
    fr_model.train()
    optimizer.zero_grad()

    YtL_pred_fr = fr_model(MR_input)
    loss = F.mse_loss(YtL_pred_fr, Ytl_tensor)

    loss.backward()
    optimizer.step()

print(f"Final loss: {loss.item():.4f}")

Final loss: 4132.4351


In [37]:
torch.save(fr_model.state_dict(), "mr_model.pth")

In [38]:
Xt_train, Xt_val, Yt_train, Yt_val = train_test_split(Xt_L.values, Yt_L.values, test_size=0.2, random_state=42)

Xt_val_tensor = torch.tensor(Xt_val, dtype=torch.float32)
Yt_val_tensor = torch.tensor(Yt_val.reshape(-1, 1), dtype=torch.float32)

Xt_train_tensor = torch.tensor(Xt_train, dtype=torch.float32)
Yt_train_tensor = torch.tensor(Yt_train.reshape(-1, 1), dtype=torch.float32)


In [39]:
def predict_with_modelX(ma_model, fs_model, fr_model, Xt_val_tensor, Yt_val_tensor):
    with torch.no_grad():
        Zt_val = ma_model(Xt_val_tensor)              
        Yt_pred_ma = fs_model(Zt_val)                 
        L_MA = F.mse_loss(Yt_pred_ma, Yt_val_tensor)  

        MR_val_input = torch.cat([Xt_val_tensor, Yt_pred_ma], dim=1)
        Yt_pred_mr = fr_model(MR_val_input)           
        L_MR = F.mse_loss(Yt_pred_mr, Yt_val_tensor)  

        print(f"Validation Loss (MA): {L_MA.item():.4f}")
        print(f"Validation Loss (MR): {L_MR.item():.4f}")


        if L_MA.item() <= L_MR.item():
            print(" MA selected. ")
            return Yt_pred_ma, 'MA'
        else:
            print(" MR selected. ")
            return Yt_pred_mr, 'MR'


In [40]:
Yt_val_pred, selected_model = predict_with_modelX(ma_model, fs_model, fr_model, Xt_val_tensor, Yt_val_tensor)

Validation Loss (MA): 5485.5317
Validation Loss (MR): 5396.4722
 MR selected. 


In [42]:
from torch.utils.data import DataLoader, TensorDataset

In [43]:
train_dataset = TensorDataset(Xtl_tensor, Ytl_tensor)
val_dataset = TensorDataset(Xt_val_tensor, Yt_val_tensor)

In [44]:
def objective(trial):
    hidden1 = trial.suggest_int("hidden1", 64, 256)
    hidden2 = trial.suggest_int("hidden2", 32, 128)
    hidden3 = trial.suggest_int("hidden3", 16, 64)
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])
    n_epochs = trial.suggest_int("n_epochs", 50, 200)

    model = nn.Sequential(
        nn.Linear(Xtl_tensor.shape[1], hidden1),
        nn.ReLU(),
        nn.Linear(hidden1, hidden2),
        nn.ReLU(),
        nn.Linear(hidden2, hidden3),
        nn.ReLU(),
        nn.Linear(hidden3, 1)
    )

    loss_fn = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    batch_start = torch.arange(0, len(Xtl_tensor), batch_size)


    model.eval()
    with torch.no_grad():
        init_pred = model(Xt_val_tensor)
        best_mse = loss_fn(init_pred, Yt_val_tensor).item()

        if np.isnan(best_mse) or np.isinf(best_mse):
            return float("inf")

        best_weights = copy.deepcopy(model.state_dict())


    for epoch in range(n_epochs):
        model.train()
        for start in batch_start:
            X_batch = Xtl_tensor[start:start+batch_size]
            y_batch = Ytl_tensor[start:start+batch_size]

            y_pred = model(X_batch)
            loss = loss_fn(y_pred, y_batch)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            y_pred = model(Xt_val_tensor)
            mse = loss_fn(y_pred, Yt_val_tensor).item()

            if not np.isnan(mse) and mse < best_mse:
                best_mse = mse
                best_weights = copy.deepcopy(model.state_dict())

    if best_weights is not None:
        model.load_state_dict(best_weights)

    if np.isnan(best_mse) or np.isinf(best_mse):
        return float("inf")

    return best_mse


In [45]:
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=30, timeout=600)
print("Best hyperparameters: ")
print(study.best_params)
print(f"Best MSE loss: {study.best_value:.4f}")

[I 2025-09-09 16:48:45,059] A new study created in memory with name: no-name-31290952-760c-496d-a868-91bd8c2be0fa
[I 2025-09-09 16:49:06,120] Trial 0 finished with value: 5888.40478515625 and parameters: {'hidden1': 129, 'hidden2': 89, 'hidden3': 23, 'lr': 0.0007895226463712642, 'batch_size': 128, 'n_epochs': 136}. Best is trial 0 with value: 5888.40478515625.
[I 2025-09-09 16:49:19,927] Trial 1 finished with value: 6081.87158203125 and parameters: {'hidden1': 153, 'hidden2': 117, 'hidden3': 30, 'lr': 0.0010833030350311398, 'batch_size': 128, 'n_epochs': 82}. Best is trial 0 with value: 5888.40478515625.
[I 2025-09-09 16:50:09,817] Trial 2 finished with value: 5672.3427734375 and parameters: {'hidden1': 170, 'hidden2': 44, 'hidden3': 58, 'lr': 0.0010919881576846007, 'batch_size': 32, 'n_epochs': 124}. Best is trial 2 with value: 5672.3427734375.
[I 2025-09-09 16:50:45,333] Trial 3 finished with value: 5950.63427734375 and parameters: {'hidden1': 189, 'hidden2': 46, 'hidden3': 37, 'lr':

Best hyperparameters: 
{'hidden1': 198, 'hidden2': 44, 'hidden3': 54, 'lr': 0.001398000011733847, 'batch_size': 32, 'n_epochs': 170}
Best MSE loss: 5627.3652


In [46]:
best_params = study.best_params

hidden1 = best_params["hidden1"]
hidden2 = best_params["hidden2"]
hidden3 = best_params["hidden3"]
lr = best_params["lr"]
batch_size = best_params["batch_size"]
n_epochs = best_params["n_epochs"]

retrain_model = nn.Sequential(
    nn.Linear(Xtl_tensor.shape[1], hidden1),
    nn.ReLU(),
    nn.Linear(hidden1, hidden2),
    nn.ReLU(),
    nn.Linear(hidden2, hidden3),
    nn.ReLU(),
    nn.Linear(hidden3, 1)
)

loss_fn = nn.MSELoss()
optimizer = optim.Adam(retrain_model.parameters(), lr=lr)
batch_start = torch.arange(0, len(Xtl_tensor), batch_size)


best_mse = float("inf")
best_weights = None

for epoch in range(n_epochs):
    retrain_model.train()
    for start in batch_start:
        X_batch = Xtl_tensor[start:start+batch_size]
        y_batch = Ytl_tensor[start:start+batch_size]

        y_pred = retrain_model(X_batch)
        loss = loss_fn(y_pred, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    retrain_model.eval()
    with torch.no_grad():
        y_val_pred = retrain_model(Xt_val_tensor)
        mse = loss_fn(y_val_pred, Yt_val_tensor).item()

        if mse < best_mse:
            best_mse = mse
            best_weights = copy.deepcopy(retrain_model.state_dict())


retrain_model.load_state_dict(best_weights)


<All keys matched successfully>

In [47]:
with torch.no_grad():
    y_val_pred = retrain_model(Xt_val_tensor)
    final_mse = loss_fn(y_val_pred, Yt_val_tensor).item()

print(f"Final validation MSE (retrained from scratch): {final_mse:.4f}")

Final validation MSE (retrained from scratch): 5811.8379


In [48]:
with torch.no_grad():
    Zt_train = ma_model(Xt_train_tensor)
    Zt_val = ma_model(Xt_val_tensor)

In [49]:
def objective_linear_probing(trial):
    lr = trial.suggest_float("lr", 1e-4, 1e-1, log=True)
    n_epochs = trial.suggest_int("n_epochs", 50, 200)
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])

    model = nn.Linear(Zt_train.shape[1], 1)  
    loss_fn = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    batch_start = torch.arange(0, len(Zt_train), batch_size)

    best_mse = float("inf")
    best_weights = None

    for epoch in range(n_epochs):
        model.train()
        for start in batch_start:
            X_batch = Zt_train[start:start+batch_size]
            y_batch = Yt_train_tensor[start:start+batch_size]

            y_pred = model(X_batch)
            loss = loss_fn(y_pred, y_batch)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            y_pred = model(Zt_val)
            mse = loss_fn(y_pred, Yt_val_tensor).item()

            if mse < best_mse:
                best_mse = mse
                best_weights = copy.deepcopy(model.state_dict())

    if best_weights is not None:
        model.load_state_dict(best_weights)

    return best_mse


In [50]:
study = optuna.create_study(direction="minimize")
study.optimize(objective_linear_probing, n_trials=30, timeout=600)

[I 2025-09-09 17:18:10,218] A new study created in memory with name: no-name-808a1075-20b8-4c61-b56e-9fca619701c1
[I 2025-09-09 17:18:11,942] Trial 0 finished with value: 67275.90625 and parameters: {'lr': 0.0001906360699694176, 'n_epochs': 168, 'batch_size': 128}. Best is trial 0 with value: 67275.90625.
[I 2025-09-09 17:18:12,855] Trial 1 finished with value: 8197.1728515625 and parameters: {'lr': 0.025843239879950117, 'n_epochs': 92, 'batch_size': 128}. Best is trial 1 with value: 8197.1728515625.
[I 2025-09-09 17:18:14,206] Trial 2 finished with value: 43939.1953125 and parameters: {'lr': 0.0024822265495446525, 'n_epochs': 138, 'batch_size': 128}. Best is trial 1 with value: 8197.1728515625.
[I 2025-09-09 17:18:15,941] Trial 3 finished with value: 42161.875 and parameters: {'lr': 0.002096146176284405, 'n_epochs': 177, 'batch_size': 128}. Best is trial 1 with value: 8197.1728515625.
[I 2025-09-09 17:18:19,241] Trial 4 finished with value: 5928.8818359375 and parameters: {'lr': 0.040

In [51]:
print("Best hyperparameters: ")
print(study.best_params)
print(f"Best MSE loss: {study.best_value:.4f}")

Best hyperparameters: 
{'lr': 0.09791631227421432, 'n_epochs': 195, 'batch_size': 64}
Best MSE loss: 5692.9956


In [54]:
best_params = study.best_params

linear_model = nn.Linear(Zt_train.shape[1], 1)
optimizer = optim.Adam(linear_model.parameters(), lr=best_params['lr'])
loss_fn = nn.MSELoss()

batch_size = best_params['batch_size']
n_epochs = best_params['n_epochs']
batch_start = torch.arange(0, len(Zt_train), batch_size)

best_mse = float('inf')
best_weights = None

for epoch in range(n_epochs):
    linear_model.train()
    for start in batch_start:
        X_batch = Zt_train[start:start+batch_size]
        y_batch = Yt_train_tensor[start:start+batch_size]

        y_pred = linear_model(X_batch)
        loss = loss_fn(y_pred, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    linear_model.eval()
    with torch.no_grad():
        y_val_pred_lp = linear_model(Zt_val)
        val_mse = loss_fn(y_val_pred_lp, Yt_val_tensor).item()
        if val_mse < best_mse:
            best_mse = val_mse
            best_weights = copy.deepcopy(linear_model.state_dict())

if best_weights is not None:
    linear_model.load_state_dict(best_weights)


In [55]:
linear_model.eval()
with torch.no_grad():
    y_val_pred_lp = linear_model(Zt_val)
    mse_lp = loss_fn(y_val_pred_lp, Yt_val_tensor).item()

print(f"Linear Probing Validation MSE: {mse_lp:.4f}")

Linear Probing Validation MSE: 5693.0044
